In [ ]:
import torch,tqdm
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import DataLoader, TensorDataset
from google.colab import drive

drive.mount('/content/drive')
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "/content/drive/MyDrive/pythia160m_pubmed",
    dtype=torch.bfloat16
)
model = model.to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(
    "/content/drive/MyDrive/pythia160m_pubmed"
)
tokenizer.pad_token = tokenizer.eos_token
print("Fine-tuned model loaded.")

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
all_activations = []

def hook_fn(module, input, output):
    hidden = output[0].detach().cpu()

    all_activations.append(hidden.reshape(-1, 768))

hook_handle = model.gpt_neox.layers[6].register_forward_hook(hook_fn)

with torch.no_grad():
    for i in tqdm(range(5000)):
        text = wiki_dataset[i]["text"]
        if len(text.strip()) == 0:
            continue
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=128
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        model(**inputs)

hook_handle.remove()
layer._forward_hooks.clear()  # clear again after

print("Unique dims:", set(a.dim() for a in all_activations))
activations_ft = torch.cat(all_activations, dim=0)
print("Shape:", activations_ft.shape)

In [ ]:
torch.save(
    activations_ft,
    "/content/drive/MyDrive/activations_finetuned.pt"
)
print("Saved.")

In [ ]:

for i, a in enumerate(all_activations[:5]):
    print(f"Tensor {i}: shape={a.shape}, dims={a.dim()}")

In [ ]:
class SparseAutoencoder(nn.Module):

    def __init__(
        self,
        input_dim=768,
        dict_size=6144,
        k=30
    ):
        super().__init__()

        self.k = k

        self.encoder = nn.Linear(input_dim, dict_size)
        self.decoder = nn.Linear(dict_size, input_dim, bias=False)
        self.pre_bias = nn.Parameter(torch.zeros(input_dim))

    @torch.no_grad()
    def normalize_decoder(self):
        norms = self.decoder.weight.data.norm(dim=0, keepdim=True)
        self.decoder.weight.data = (
            self.decoder.weight.data / norms.clamp(min=1e-8)
        )

    def forward(self, x):
        x_centered = x - self.pre_bias
        pre_activations = self.encoder(x_centered)
        topk_vals, topk_idx = torch.topk(pre_activations, self.k, dim=-1)
        features = torch.zeros_like(pre_activations)
        features.scatter_(-1, topk_idx, torch.relu(topk_vals))
        reconstruction = self.decoder(features) + self.pre_bias
        return reconstruction, features

In [ ]:
sae2 = SparseAutoencoder(
    input_dim=768,
    dict_size=6144,
    k=30
).to(device)

optimizer2 = torch.optim.Adam(sae2.parameters(), lr=1e-4)

scheduler2 = torch.optim.lr_scheduler.LinearLR(
    optimizer2,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=500
)
print("SAE 2 initialized.")

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dataset_ft = TensorDataset(activations_ft.float())
loader_ft = DataLoader(dataset_ft, batch_size=512, shuffle=True)

sae2 = SparseAutoencoder(
    input_dim=768,
    dict_size=6144,
    k=30
).to(device)

optimizer2 = torch.optim.Adam(sae2.parameters(), lr=1e-4)
scheduler2 = torch.optim.lr_scheduler.LinearLR(
    optimizer2,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=500
)

epochs = 10
for epoch in range(epochs):
    total_loss = 0
    for batch in loader_ft:
        x = batch[0].to(device)
        reconstruction, features = sae2(x)
        loss = ((x - reconstruction) ** 2).mean()
        optimizer2.zero_grad()
        loss.backward()
        optimizer2.step()
        sae2.normalize_decoder()
        scheduler2.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} | Loss={total_loss/len(loader_ft):.4f}")

In [ ]:
# BY MISTAKE DELETED THE CELL IN WHICH TRAINED THE MODEL FOR 10 EPOCHS MORE AND LOG WAS Epoch 1 | Loss=0.0674
Epoch 2 | Loss=0.0667
Epoch 3 | Loss=0.0660
Epoch 4 | Loss=0.0654
Epoch 5 | Loss=0.0648
Epoch 6 | Loss=0.0643
Epoch 7 | Loss=0.0638
Epoch 8 | Loss=0.0634
Epoch 9 | Loss=0.0630
Epoch 10 | Loss=0.0626

In [ ]:
with torch.no_grad():
    sample = activations_ft[:1000].float().to(device)
    _, features = sae2(sample)
    active_fraction = (features > 0).float().mean()
    print("Active Fraction:", active_fraction.item())
    print("Expected:", 30/6144)

In [ ]:
torch.save(
    sae2.state_dict(),
    "/content/drive/MyDrive/sae_layer6_finetuned_v2.pt"
)
print("SAE 2 saved.")